# Lab 12: Faster R-CNN Inference

**Module 12** | Read `notes/12-faster-rcnn.pdf` before running this notebook.

Run a pre-trained Faster R-CNN on sample images, filter detections by score, and save annotated outputs.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Faster R-CNN object detection

Faster R-CNN is a two-stage detector: a backbone extracts features, a Region Proposal Network suggests candidate boxes, and a classification head refines each region. Torchvision ships a ResNet-50 FPN variant pre-trained on COCO.

This lab loads those weights, runs inference on local sample images, keeps detections with confidence above 0.5, and saves annotated copies for inspection.


### Load the pre-trained model

`FasterRCNN_ResNet50_FPN_Weights.DEFAULT` bundles both network weights and the preprocessing transforms (resize, normalize) expected at inference time.


In [ ]:
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)

ROOT = Path("..").resolve()
IMAGE_DIR = ROOT / "datasets" / "sample_images"
OUT_DIR = ROOT / "labs" / "annotated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval().to(device)
preprocess = weights.transforms()
category_names = weights.meta["categories"]

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(image_paths)} images in {IMAGE_DIR}")
if not image_paths:
    raise FileNotFoundError(
        f"No JPG files in {IMAGE_DIR}. Run: python scripts/download_datasets.py"
    )


### Run inference and draw boxes

The model returns `boxes`, `labels`, and `scores` per image. We filter by `score > 0.5`, draw rectangles with class names, and write PNG files to `labs/annotated/`.


In [ ]:
SCORE_THRESHOLD = 0.5
saved: list[Path] = []

try:
    font = ImageFont.truetype("arial.ttf", 16)
except OSError:
    font = ImageFont.load_default()

for img_path in image_paths:
    image = Image.open(img_path).convert("RGB")
    tensor = preprocess(image).to(device)
    with torch.no_grad():
        prediction = model([tensor])[0]

    draw = ImageDraw.Draw(image)
    kept = 0
    for box, label, score in zip(
        prediction["boxes"], prediction["labels"], prediction["scores"]
    ):
        if score.item() <= SCORE_THRESHOLD:
            continue
        kept += 1
        x1, y1, x2, y2 = box.tolist()
        name = category_names[label.item()]
        draw.rectangle([x1, y1, x2, y2], outline="lime", width=3)
        draw.text((x1, max(y1 - 18, 0)), f"{name} {score:.2f}", fill="yellow", font=font)

    out_path = OUT_DIR / f"{img_path.stem}_det.jpg"
    image.save(out_path)
    saved.append(out_path)
    print(f"{img_path.name}: {kept} detections (score > {SCORE_THRESHOLD}) -> {out_path.name}")

print(f"\nSaved {len(saved)} annotated images to {OUT_DIR}")


### Preview one result inline

Open the first annotated image in the notebook to verify labels and box placement.


In [ ]:
import matplotlib.pyplot as plt

preview = Image.open(saved[0])
plt.figure(figsize=(10, 7))
plt.imshow(preview)
plt.axis("off")
plt.title(saved[0].name)
plt.show()
